# TESTING

## Prueba del LocalTunnel
- Devuelve siempre {"estado": "activo"}
- https://pixmindworkervehicle.loca.lt/pixmindWorkerVehicle en POST con una imagen en clave imagen tipo File.

In [ ]:
# Instalamos flask y localtunnel
!pip install flask flask-ngrok --quiet
!npm install -g localtunnel

from flask import Flask, request, jsonify
import subprocess
import threading


app = Flask(__name__)

@app.route("/pixmindWorkerVehicle", methods=["POST"])
def pixmind_worker_vehicle():
    if "imagen" not in request.files:
        return jsonify({"error": "falta el archivo 'imagen'"}), 400
    # Se recibe la imagen pero no se procesa
    return jsonify({"estado": "activo"})

# Función para ejecutar localtunnel en un hilo aparte
def run_localtunnel():
    # El puerto debe coincidir con el de Flask
    cmd = "lt --port 5000 --subdomain pixmindworkervehicle"
    subprocess.call(cmd, shell=True)

# Ejecutar localtunnel en segundo plano
threading.Thread(target=run_localtunnel).start()

# Ejecutar Flask
app.run(port=5000)

⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 1s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇ * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [22/Nov/2025 14:47:57] "POST /pixmindWorkerVehicle HTTP/1.1" 200 -


## Prueba Detector de vehiculos
- devuelve un json con informacion del tipo de vehiculo, confianza, cantidad de vehiculos detectados y detalles de la CV
- - https://pixmindworkervehicle.loca.lt/pixmindWorkerVehicle en POST con una imagen en clave imagen tipo File.

In [ ]:
# 1. INSTALAMOS DEPENDENCIAS (Agregamos ultralytics y pillow)
!pip install flask flask-ngrok ultralytics pillow --quiet
!npm install -g localtunnel

from flask import Flask, request, jsonify
from ultralytics import YOLO
from PIL import Image
import subprocess
import threading
import io

""" DEFINIMOS EL MODELO DEEP LEARNING """
# Cargamos YOLOv8 Nano (se descargará automáticamente la primera vez)
print("Cargando modelo YOLOv8 Nano...")
model = YOLO('yolov8n.pt')

# Definimos los IDs de clases de vehículos en el dataset COCO
# 2: car, 3: motorcycle, 5: bus, 7: truck
VEHICLE_CLASSES = [2, 3, 5, 7]

""" DEFINIMOS EL SERVICIO WEB FLASK """

app = Flask(__name__)

@app.route("/pixmindVehicle", methods=["POST"])
def pixmind_worker_vehicle():
    if "imagen" not in request.files:
        return jsonify({"error": "falta el archivo 'imagen'"}), 400

    try:
        # 1. Leemos la imagen directamente de la memoria
        archivo = request.files['imagen']
        img = Image.open(archivo.stream)

        # 2. Hacemos la inferencia
        # conf=0.4 descarta predicciones con menos del 40% de seguridad
        results = model(img, conf=0.4, verbose=False)

        detecciones = []
        vehiculo_encontrado = False

        # 3. Procesamos los resultados
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls[0]) # ID de la clase detectada

                # Verificamos si es un vehículo
                if cls_id in VEHICLE_CLASSES:
                    vehiculo_encontrado = True
                    nombre_clase = model.names[cls_id] # Ej: 'car', 'truck'
                    confianza = float(box.conf[0])

                    detecciones.append({
                        "tipo": nombre_clase,
                        "confianza": round(confianza, 2)
                    })

        # 4. Retornamos la respuesta JSON
        return jsonify({
            "vehiculo_detectado": vehiculo_encontrado,
            "cantidad": len(detecciones),
            "detalles": detecciones
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Función para ejecutar localtunnel
def run_localtunnel():
    # El puerto debe coincidir con el de Flask
    cmd = ENVlocaltunnel
    subprocess.call(cmd, shell=True)

# Ejecutar localtunnel en segundo plano
threading.Thread(target=run_localtunnel).start()

# Ejecutar Flask
print("Servidor iniciando en puerto 5000...")
app.run(port=5000)

⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏Cargando modelo YOLOv8 Nano...
Servidor iniciando en puerto 5000...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [22/Nov/2025 21:22:51] "POST /pixmindVehicle HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Nov/2025 21:23:00] "POST /pixmindVehicle HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Nov/2025 21:23:07] "POST /pixmindVehicle HTTP/1.1" 200 -


## Prueba más detalles YOLO
Al usar YOLO le añadimos mas detalles de la vision. Pösiciión, dimenciones, lugar en la imagen, dimension

In [ ]:
from flask import Flask, request, jsonify
from ultralytics import YOLO
from PIL import Image
import subprocess
import threading
import io

""" DEFINIMOS EL MODELO DEEP LEARNING """
# Cargamos el modelo ligero
model = YOLO('yolov8n.pt')

# IDs COCO: 2=car, 3=motorcycle, 5=bus, 7=truck
VEHICLE_CLASSES = [2, 3, 5, 7]

""" FUNCIONES AUXILIARES """

def obtener_color_predominante(img_original, coordenadas):
    """
    Recorta el vehículo y detecta el color promedio.
    Retorna un string hexadecimal (ej: #FF0000).
    """
    try:
        x1, y1, x2, y2 = map(int, coordenadas)
        # Recortamos solo la parte de la imagen donde está el vehículo
        recorte = img_original.crop((x1, y1, x2, y2))
        # Reducimos a 1 pixel para promediar el color (truco rápido)
        color_pixel = recorte.resize((1, 1)).getpixel((0, 0))
        # Convertimos RGB a Hexadecimal
        return '#{:02x}{:02x}{:02x}'.format(*color_pixel)
    except:
        return "#000000"

""" DEFINIMOS EL SERVICIO WEB FLASK """

app = Flask(__name__)

@app.route("/pixmindWorkerVehicle", methods=["POST"])
def pixmind_worker_vehicle():
    if "imagen" not in request.files:
        return jsonify({"error": "falta el archivo 'imagen'"}), 400

    try:
        archivo = request.files['imagen']
        img = Image.open(archivo.stream).convert("RGB") # Aseguramos RGB
        width, height = img.size

        # Inferencia
        results = model(img, conf=0.4, verbose=False)

        detecciones = []
        vehiculo_encontrado = False

        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls[0])

                if cls_id in VEHICLE_CLASSES:
                    vehiculo_encontrado = True
                    nombre_clase = model.names[cls_id]
                    confianza = float(box.conf[0])

                    # Obtenemos coordenadas [x1, y1, x2, y2]
                    coords = box.xyxy[0].tolist()

                    # Calculamos dimensiones del vehículo
                    ancho_vehiculo = coords[2] - coords[0]
                    alto_vehiculo = coords[3] - coords[1]

                    # Extraemos el color
                    color_hex = obtener_color_predominante(img, coords)

                    detecciones.append({
                        "tipo": nombre_clase,
                        "confianza": round(confianza, 2),
                        "color_hex": color_hex,
                        "posicion": {
                            "x1": int(coords[0]),
                            "y1": int(coords[1]),
                            "x2": int(coords[2]),
                            "y2": int(coords[3])
                        },
                        "dimensiones": {
                            "ancho": int(ancho_vehiculo),
                            "alto": int(alto_vehiculo)
                        }
                    })

        return jsonify({
            "meta": {
                "ancho_imagen": width,
                "alto_imagen": height
            },
            "vehiculo_detectado": vehiculo_encontrado,
            "total_vehiculos": len(detecciones),
            "detalles": detecciones
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Función LocalTunnel
def run_localtunnel():
    cmd = "lt --port 5000 --subdomain pixmindworkervehicle"
    subprocess.call(cmd, shell=True)

# Hilos y Ejecución
threading.Thread(target=run_localtunnel).start()
print("Servidor mejorado iniciando en puerto 5000...")
app.run(port=5000)

## Prueba OCR detector de placas
- Se le añade un lector de placas. Es mucho mas lento el servicio.

- https://pixmindworkervehicle.loca.lt/pixmindWorkerVehicle en POST con una imagen en clave imagen tipo File.

## Prueba detector avanzado
* Mejor predicion de placas
* soporte con imagen

In [ ]:
# ===============================================
# CONFIGURACIÓN E INSTALACIONES CRÍTICAS
# ===============================================

# 1. INSTALAMOS DEPENDENCIAS (Incluyendo OpenCV y las necesarias para Flask)
!pip install flask ultralytics pillow easyocr opencv-python-headless numpy --quiet
!npm install -g localtunnel

# ===============================================
# 1. IMPORTS
# ===============================================
from flask import Flask, request, jsonify, send_file
from ultralytics import YOLO
from PIL import Image
import easyocr
import numpy as np
import subprocess
import threading
import io
import cv2
import torch
import math

# ===============================================
# 2. CARGAMOS LOS MODELOS Y CONFIGURACIÓN GLOBAL
# ===============================================

# 2.1 Configuración de Clases y Umbrales
VEHICLE_CLASSES_YOLO = [2, 3, 5, 7]  # Car, Motorcycle, Bus, Truck (Clases COCO para YOLOv8)
VEHICLE_CONF_THRESHOLD = 0.50
OCR_CONF_THRESHOLD = 0.25

# 2.2 Zonas Heurísticas (Reemplazo robusto para LPD)
ZONAS_HEURISTICAS_AGRESIVAS = {
    "Zona Amplia Inferior": (0.10, 0.60, 0.90, 0.95),  # (x_min, y_min, x_max, y_max)
    "Centro Estándar": (0.30, 0.70, 0.70, 0.95),
}

# 2.3 Carga de Modelos
print("Cargando YOLOv8 Nano...")
model = YOLO('yolov8n.pt')

print("Cargando EasyOCR...")
# Usar gpu=True si detecta CUDA o False por defecto para compatibilidad en Colab Free
READER = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("✅ Modelos y Reader inicializados.")

# ===============================================
# 3. FUNCIONES AUXILIARES DE PLACA Y COLOR
# ===============================================

def es_formato_placa_colombiana(texto):
    """Verifica el formato LLLNNN con tolerancia."""
    texto = texto.replace(' ', '').replace('-', '').upper()
    if len(texto) != 6: return False
    letras = texto[:3]
    numeros = texto[3:]
    return letras.isalpha() and numeros.isdigit()

def obtener_color_hex(img_recorte_pil):
    """Calcula el color promedio en Hexadecimal a partir de una imagen PIL."""
    try:
        # Convertir a numpy para manejarlo con OpenCV si es necesario, pero PIL es suficiente aquí
        img_small = img_recorte_pil.resize((1, 1))
        color = img_small.getpixel((0, 0))
        # Asegurarse de que el color sea un RGB tuple
        if isinstance(color, int): # Si solo es un int (imagen en escala de grises), lo forzamos
             color = (color, color, color)
        return '#{:02x}{:02x}{:02x}'.format(color[0], color[1], color[2])
    except:
        return "#000000"

def detectar_y_leer_placa(img_vehiculo_np, reader):
    """
    Aplica la Multi-Heurística Agresiva y el OCR robusto sobre el recorte del vehículo.
    img_vehiculo_np debe ser un array de numpy (BGR).
    """
    if reader is None: return None

    h_v, w_v, _ = img_vehiculo_np.shape
    mejor_placa = None

    for nombre_zona, (x_min_p, y_min_p, x_max_p, y_max_p) in ZONAS_HEURISTICAS_AGRESIVAS.items():
        # Calcular coordenadas locales
        x1_p_local = int(w_v * x_min_p)
        y1_p_local = int(h_v * y_min_p)
        x2_p_local = int(w_v * x_max_p)
        y2_p_local = int(h_v * y_max_p)

        # Recorte de la placa en coordenadas LOCALES (numpy slice)
        recorte_placa_local = img_vehiculo_np[y1_p_local:y2_p_local, x1_p_local:x2_p_local].copy()

        if recorte_placa_local.size == 0: continue

        # OCR ROBUSTO
        mejor_texto_ocr = None
        mejor_conf_ocr = 0.0

        crops_to_test = [recorte_placa_local, cv2.cvtColor(recorte_placa_local, cv2.COLOR_BGR2GRAY)]

        for crop in crops_to_test:
             current_results = reader.readtext(
                crop,
                allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                width_ths=0.7,
                paragraph=False
            )

             for (bbox, text, conf) in current_results:
                texto_limpio = text.replace(' ', '').replace('-', '').upper()

                # CRITERIO DE ROBUSTEZ 1: Formato válido Y Confianza > Umbral
                if es_formato_placa_colombiana(texto_limpio) and conf > OCR_CONF_THRESHOLD:
                    if conf > mejor_conf_ocr:
                        mejor_conf_ocr = conf
                        mejor_texto_ocr = texto_limpio
                        if conf >= 0.95: break

                # CRITERIO DE ROBUSTEZ 2: Mejor lectura general y es larga
                elif conf > mejor_conf_ocr and len(texto_limpio) >= 6:
                     mejor_conf_ocr = conf
                     mejor_texto_ocr = texto_limpio

        # Si encontramos una placa mejor que la anterior, la guardamos
        if mejor_texto_ocr:
            if mejor_placa is None or mejor_conf_ocr > mejor_placa['confianza']:
                mejor_placa = {
                    "texto": mejor_texto_ocr,
                    "confianza": round(mejor_conf_ocr, 4),
                    # Coordenadas de la placa RELATIVAS al recorte del vehículo
                    "posicion_relativa_vehiculo": [x1_p_local, y1_p_local, x2_p_local, y2_p_local],
                    "zona_heuristica": nombre_zona
                }

    return mejor_placa # Devuelve None o el diccionario con la mejor placa


# ===============================================
# 4. SERVICIO WEB FLASK
# ===============================================

app = Flask(__name__)

@app.route("/pixmindVehicle", methods=["POST"])
def pixmind_worker_vehicle():
    if "imagen" not in request.files:
        return jsonify({"error": "falta el archivo 'imagen'. Asegúrate de enviarla como form-data con la key 'imagen'."}), 400

    try:
        # Lee la imagen enviada
        archivo = request.files['imagen']
        img_pil = Image.open(archivo.stream).convert("RGB")
        width, height = img_pil.size

        # Convertir PIL a NumPy (OpenCV) para dibujar y hacer los recortes
        img_np = np.array(img_pil)
        # OpenCV usa formato BGR por defecto
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        img_vis = img_bgr.copy() # Copia para dibujar las cajas

        # -----------------------------
        # Detección de Vehículos (YOLOv8)
        # -----------------------------
        results = model(img_pil, conf=VEHICLE_CONF_THRESHOLD, classes=VEHICLE_CLASSES_YOLO, verbose=False)

        detecciones = []
        vehiculos_con_placa = 0

        # Recorrer resultados
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls[0])

                # Filtrado de vehículo (redundante si se usó classes=)
                if cls_id in VEHICLE_CLASSES_YOLO:

                    nombre_clase = model.names[cls_id]
                    conf_vehiculo = float(box.conf[0])
                    coords_global = list(map(int, box.xyxy[0])) # [x1, y1, x2, y2]
                    x1_v, y1_v, x2_v, y2_v = coords_global

                    # 1. Recorte y Color
                    recorte_vehiculo_pil = img_pil.crop((x1_v, y1_v, x2_v, y2_v))
                    recorte_vehiculo_np = img_bgr[y1_v:y2_v, x1_v:x2_v].copy()
                    color_hex = obtener_color_hex(recorte_vehiculo_pil)

                    # 2. Lectura de Placa (Multi-Heurística Robusta)
                    datos_placa = None
                    if recorte_vehiculo_np.size > 0 and recorte_vehiculo_np.shape[0] > 30: # Evita recortes vacíos/pequeños
                        datos_placa_raw = detectar_y_leer_placa(recorte_vehiculo_np, READER)

                        if datos_placa_raw:
                            # Calculamos la posición de la placa en coordenadas GLOBALES
                            pr_x1, pr_y1, pr_x2, pr_y2 = datos_placa_raw['posicion_relativa_vehiculo']

                            datos_placa = {
                                "texto": datos_placa_raw['texto'],
                                "confianza": datos_placa_raw['confianza'],
                                "posicion_global": [
                                    x1_v + pr_x1, y1_v + pr_y1,
                                    x1_v + pr_x2, y1_v + pr_y2
                                ]
                            }

                            vehiculos_con_placa += 1

                            # 3. DIBUJAR LA PLACA EN LA IMAGEN DE RESPUESTA (ROJO BGR)
                            pl_x1, pl_y1, pl_x2, pl_y2 = datos_placa['posicion_global']
                            cv2.rectangle(img_vis, (pl_x1, pl_y1), (pl_x2, pl_y2), (0, 0, 255), 2)
                            cv2.putText(img_vis, datos_placa['texto'], (pl_x1, pl_y1 - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                    # 4. DIBUJAR EL VEHÍCULO EN LA IMAGEN DE RESPUESTA (VERDE BGR)
                    cv2.rectangle(img_vis, (x1_v, y1_v), (x2_v, y2_v), (0, 255, 0), 2)

                    placa_texto = datos_placa['texto'] if datos_placa else "N/A"
                    texto_v = f"{nombre_clase} {conf_vehiculo:.2f} | Placa:{placa_texto}"
                    cv2.putText(img_vis, texto_v, (x1_v, y1_v - 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)


                    # 5. Armamos el objeto de entrega JSON
                    detecciones.append({
                        "tipo": nombre_clase,
                        "confianza_vehiculo": round(conf_vehiculo, 2),
                        "color_hex": color_hex,
                        "placa": datos_placa,
                        "posicion_vehiculo_global": coords_global
                    })

        # -----------------------------
        # Preparar y Enviar la Respuesta
        # -----------------------------

        # Convertir la imagen final de OpenCV (BGR) a JPEG en memoria
        is_success, buffer = cv2.imencode(".jpg", img_vis)
        if not is_success:
            raise Exception("Error al codificar la imagen procesada a JPEG.")

        byte_io = io.BytesIO(buffer)
        byte_io.seek(0)

        # Usamos send_file para enviar la imagen binaria directamente
        return send_file(byte_io, mimetype='image/jpeg', as_attachment=False, download_name='deteccion_vehiculos_placa.jpg')

    except Exception as e:
        # Devuelve un JSON con el error si algo falla
        return jsonify({"error": str(e), "message": "Fallo durante el procesamiento de la imagen o la detección."}), 500

# Función LocalTunnel y Arranque
def run_localtunnel():
    cmd = ENVlocaltunnel
    # Usamos try/except para capturar si localtunnel no funciona correctamente
    try:
        subprocess.check_call(cmd, shell=True)
    except subprocess.CalledProcessError as e:
        print(f"Error al ejecutar localtunnel: {e}")
    except FileNotFoundError:
        print("Error: localtunnel no está instalado. Asegúrate de que la instalación con npm fue exitosa.")

# Ejecutar el localtunnel en un hilo aparte
threading.Thread(target=run_localtunnel).start()
print("\n--- INICIANDO SERVICIO FLASK ---")
print("Servidor iniciando en puerto 5000...")
app.run(port=5000, use_reloader=False) # use_reloader=False es importante en entornos como Colab

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 865ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

Cargando YOLOv8 Nano...
Cargando EasyOCR...
✅ Modelos y Reader inicializados.

--- INICIANDO SERVICIO FLASK ---
Servidor iniciando en puerto 5000...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:14:14] "POST /pixmindVehicle HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:16:53] "POST /pixmindVehicle HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:17:27] "POST /pixmindVehicle HTTP/1.1" 200 -


## Prueba vehiculos con imagen de referencia (mulivehiculos)
- version OCR pero se devuevle uan imagen con anotaciones como la avanzada

In [ ]:
# 1. INSTALAMOS DEPENDENCIAS (Agregamos easyocr)
!pip install flask ultralytics pillow easyocr --quiet
!npm install -g localtunnel

from flask import Flask, request, jsonify
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont # Se importa ImageDraw y ImageFont para dibujar
import easyocr
import numpy as np
import subprocess
import threading
import io
import base64 # Se importa base64 para codificar la imagen de salida

""" 1. CARGAMOS LOS MODELOS """

# Modelo YOLO (Detección de objetos)
print("Cargando YOLOv8 Nano...")
model = YOLO('yolov8n.pt')
VEHICLE_CLASSES = [1, 2, 3, 5, 7] #Bicycle, Car, Motorcycle, Bus, Truck

# Modelo OCR (Lectura de texto)
# 'en' sirve bien para placas porque son alfanuméricas.
print("Cargando EasyOCR...")
# gpu=True si estás en Colab con T4 (se puede usar torch con torch.cuda.is_available() para detectar automaticamente)
reader = easyocr.Reader(['en'], gpu=False)

""" 2. FUNCIONES AUXILIARES """

def obtener_color_hex(img_recorte):
    """Calcula el color promedio en Hexadecimal"""
    try:
        img_small = img_recorte.resize((1, 1)) #reduce la imagen a 1 pixel
        color = img_small.getpixel((0, 0)) # Devuelve rrggbb del pixel
        return '#{:02x}{:02x}{:02x}'.format(*color) #retorna en formato hexadecial
    except:
        return "#000000" #si falla retorna negro

def leer_placa_vehiculo(img_recorte):
    """
    Recibe la imagen RECORTADA del vehículo y busca texto.
    Devuelve el texto con mayor confianza.
    """
    # Convertimos imagen PIL (formato pillow) a Array de Numpy para EasyOCR y hacerlo trabajable
    img_np = np.array(img_recorte)

    # EasyOCR devuelve una lista: [ (caja, texto, confianza), ... ]
    # allowlist: Filtramos para que solo busque letras y números estándar de placas
    result = reader.readtext(img_np, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

    detectado = False
    texto = ""
    confianza = 0.0

    # Buscamos el texto con mayor confianza que tenga al menos 4 caracteres
    best_conf = 0
    for res in result:
        text_read = res[1]
        conf = res[2]
        # Filtro simple: las placas suelen tener más de 3 o 4 caracteres
        if len(text_read) > 3 and conf > best_conf:
            best_conf = conf
            texto = text_read
            detectado = True

    if detectado:
        return {"texto": texto.upper(), "confianza": round(best_conf, 2)}
    else:
        return None

""" 3. SERVICIO WEB FLASK """

app = Flask(__name__) # se crea app Flask

# Intentamos cargar una fuente simple, si falla, se usará la fuente por defecto.
try:
    # Usamos una fuente conocida o una genérica
    font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    try:
        font = ImageFont.truetype(font_path, 20)
    except IOError:
        font = ImageFont.load_default()
except:
    font = ImageFont.load_default()


@app.route("/pixmindVehicle", methods=["POST"]) # Se crea la ruta /pixmindVehicle de tipo POST
def pixmind_worker_vehicle():
    if "imagen" not in request.files:
        return jsonify({"error": "falta el archivo 'imagen'"}), 400 # En los request files debe existir una llamada 'imagen'

    try:
        # Lee el archvio enviado FileStorage y se abre con PIL desde stream
        archivo = request.files['imagen']
        img = Image.open(archivo.stream).convert("RGB")
        width, height = img.size # se guardan alto ya ncho para metadata

        # Creamos una copia de la imagen para dibujar las anotaciones
        img_annotated = img.copy()
        draw = ImageDraw.Draw(img_annotated)

        # Detección de Vehículos
        results = model(img, conf=0.4, verbose=False) # Inferencia YOLO sobre la imagen

        detecciones = []
        vehiculos_con_placa = 0
        vehiculo_index = 1 # Contador para identificar los vehículos en la imagen

        # Recorre los resultados de YOLO
        for result in results:
            boxes = result.boxes # Lista de boxes detectadas
            # Se recorren los boxes detectados
            for box in boxes:
                cls_id = int(box.cls[0]) # indice 0: Clase detectada en formato indice (0-9000)
                # filtrado solo para clases vehiculo
                if cls_id in VEHICLE_CLASSES:
                    # Datos básicos
                    nombre_clase = model.names[cls_id] # Etiqueta textual
                    conf_vehiculo = float(box.conf[0]) # Confianza de la caja
                    coords = list(map(int, box.xyxy[0])) # coordenadas de la caja [x1, y1, x2, y2]
                    x1, y1, x2, y2 = coords

                    # Recortamos el vehículo de la imagen original (region PIL)
                    recorte_vehiculo = img.crop((x1, y1, x2, y2))

                    # Color promedio del vehiculo
                    color_hex = obtener_color_hex(recorte_vehiculo)

                    # Lectura de Placa (OCR)
                    datos_placa = None
                    if recorte_vehiculo.width > 50 and recorte_vehiculo.height > 30:
                        datos_placa = leer_placa_vehiculo(recorte_vehiculo)

                    if datos_placa:
                        vehiculos_con_placa += 1

                    # Creacion de Bounding Boxes vehiculo a vehiculo
                    draw.rectangle(coords, outline="lime", width=2) # 1. Dibujar el Bounding Box (línea verde)
                    placa_texto = datos_placa["texto"] if datos_placa else "Sin Placa" # 2. Texto a mostrar en la caja
                    texto_display = f"#{vehiculo_index} {nombre_clase} | {placa_texto}" # Se usa el índice del vehículo para referenciarlo en el JSON
                    text_bbox = draw.textbbox((x1, y1), texto_display, font=font) # 3. Dibujar fondo para el texto (para mejor legibilidad)
                    draw.rectangle([x1, text_bbox[1], text_bbox[2] + 5, text_bbox[3] + 5], fill="lime") # 4. Dibujar el texto (color negro)
                    draw.text((x1 + 2, y1 + 2), texto_display, fill="black", font=font)


                    # Armamos el objeto de entrega
                    detecciones.append({
                        "id_vehiculo": vehiculo_index,
                        "tipo": nombre_clase,
                        "confianza_vehiculo": round(conf_vehiculo, 2),
                        "color_hex": color_hex,
                        "placa": datos_placa,
                        "posicion": coords
                    })

                    vehiculo_index += 1

        # PREPARAR IMAGEN BASE64
        buffer = io.BytesIO() # Guardamos la imagen con los Bounding Boxes propios en un buffer de memoria
        img_annotated.save(buffer, format="JPEG")
        imagen_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8") # Codificamos el buffer a base64 (string) Así se reciben las imágenes en el API

        # Devuelve un JSON con la información detectada y la imagen anotada
        return jsonify({
            "meta": {"ancho": width, "alto": height},
            "total_vehiculos": len(detecciones),
            "vehiculos_con_placa_leida": vehiculos_con_placa,
            "detalles": detecciones,
            "imagen_analizada_base64": imagen_base64,
            "mimeType": "image/jpeg"
        })

# si algo falla maneja las excepciones con mensaje 500 (web, problema de servidor)
    except Exception as e:
        # Se imprime el error en la consola del servidor para depuración
        print(f"Error en pixmind_worker_vehicle: {e}")
        return jsonify({"error": str(e)}), 500

# Función LocalTunnel y Arranque
def run_localtunnel():
    cmd = ENVlocaltunnel # Se ejecuta con el nombre del subdominio definido en env.
    subprocess.call(cmd, shell=True)

# Ejecutar el localtunnel en un hilo aparte
threading.Thread(target=run_localtunnel).start()
print("Servidor iniciando en puerto 5000...")
app.run(port=5000) # Bloquea el servicio en el puerto 5000

# FINAL WORKERS

## Documentación
Los workers son servicios independientes (secretos industriales) que alimentan el servicio de PixMind. Cada servicio tiene una tarea y sus propios hiperparámetros y estructura de resultado según la naturaleza de su implementación.
### Servicio web
Para el servicio web libre y abierto, se optó por usar flask y localtunnel. Que crean el servicio de forma local y lo provicionana de un dominio temporal. El dominio se basa en "pixmindworkers" y cada endpoint será el nombre del worker. Ej: "http://www.pixmindworkers.com/vehicle"
### Dependencias
- Se requieren __Flask__, __request__ y __jsonify__ para API y manejo de peticiones / respuestas web
- __subprocess__, __threading__: laznar el __localtunnel__ en otro hilo/proceso.
- __io__: buffers en memoria
- De ultralytics __YOLO__: para cargar modelso de Yolov8
- __easyocr__: OCR orientado a texto en imágenes.
- __numpy__: para conversión PIL -> array para el EasyOCR
- __PIL.Image__: para cargar/convertir/recortar imágenes
###

In [ ]:
# ENV
ENVlocaltunnel = "lt --port 5000 --subdomain pixmindworkers"

## VEHICLE
Se usa para mostrar detalles de vehículos. Lee placas, muestra el color aproximado, soporte multivehículo.

In [ ]:
# 1. INSTALAMOS DEPENDENCIAS (Agregamos easyocr)
!pip install flask ultralytics pillow easyocr --quiet
!npm install -g localtunnel

from flask import Flask, request, jsonify
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont # Se importa ImageDraw y ImageFont para dibujar
import easyocr
import numpy as np
import subprocess
import threading
import io
import base64 # Se importa base64 para codificar la imagen de salida

""" 1. CARGAMOS LOS MODELOS """

# Modelo YOLO (Detección de objetos)
print("Cargando YOLOv8 Nano...")
model = YOLO('yolov8n.pt')
VEHICLE_CLASSES = [1, 2, 3, 5, 7] #Bicycle, Car, Motorcycle, Bus, Truck

# Modelo OCR (Lectura de texto)
# 'en' sirve bien para placas porque son alfanuméricas.
print("Cargando EasyOCR...")
# gpu=True si estás en Colab con T4 (se puede usar torch con torch.cuda.is_available() para detectar automaticamente)
reader = easyocr.Reader(['en'], gpu=False)

""" 2. FUNCIONES AUXILIARES """

def obtener_color_hex(img_recorte):
    """Calcula el color promedio en Hexadecimal"""
    try:
        img_small = img_recorte.resize((1, 1)) #reduce la imagen a 1 pixel
        color = img_small.getpixel((0, 0)) # Devuelve rrggbb del pixel
        return '#{:02x}{:02x}{:02x}'.format(*color) #retorna en formato hexadecial
    except:
        return "#000000" #si falla retorna negro

def leer_placa_vehiculo(img_recorte):
    """
    Recibe la imagen RECORTADA del vehículo y busca texto.
    Devuelve el texto con mayor confianza.
    """
    # Convertimos imagen PIL (formato pillow) a Array de Numpy para EasyOCR y hacerlo trabajable
    img_np = np.array(img_recorte)

    # EasyOCR devuelve una lista: [ (caja, texto, confianza), ... ]
    # allowlist: Filtramos para que solo busque letras y números estándar de placas
    result = reader.readtext(img_np, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

    detectado = False
    texto = ""
    confianza = 0.0

    # Buscamos el texto con mayor confianza que tenga al menos 4 caracteres
    best_conf = 0
    for res in result:
        text_read = res[1]
        conf = res[2]
        # Filtro simple: las placas suelen tener más de 3 o 4 caracteres
        if len(text_read) > 3 and conf > best_conf:
            best_conf = conf
            texto = text_read
            detectado = True

    if detectado:
        return {"texto": texto.upper(), "confianza": round(best_conf, 2)}
    else:
        return None

""" 3. SERVICIO WEB FLASK """

app = Flask(__name__) # se crea app Flask

# Intentamos cargar una fuente simple, si falla, se usará la fuente por defecto.
try:
    # Usamos una fuente conocida o una genérica
    font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    try:
        font = ImageFont.truetype(font_path, 20)
    except IOError:
        font = ImageFont.load_default()
except:
    font = ImageFont.load_default()


@app.route("/pixmindVehicle", methods=["POST"]) # Se crea la ruta /pixmindVehicle de tipo POST
def pixmind_worker_vehicle():
    if "imagen" not in request.files:
        return jsonify({"error": "falta el archivo 'imagen'"}), 400 # En los request files debe existir una llamada 'imagen'

    try:
        # Lee el archvio enviado FileStorage y se abre con PIL desde stream
        archivo = request.files['imagen']
        img = Image.open(archivo.stream).convert("RGB")
        width, height = img.size # se guardan alto ya ncho para metadata

        # Creamos una copia de la imagen para dibujar las anotaciones
        img_annotated = img.copy()
        draw = ImageDraw.Draw(img_annotated)

        # Detección de Vehículos
        results = model(img, conf=0.4, verbose=False) # Inferencia YOLO sobre la imagen

        detecciones = []
        vehiculos_con_placa = 0
        vehiculo_index = 1 # Contador para identificar los vehículos en la imagen

        # Recorre los resultados de YOLO
        for result in results:
            boxes = result.boxes # Lista de boxes detectadas
            # Se recorren los boxes detectados
            for box in boxes:
                cls_id = int(box.cls[0]) # indice 0: Clase detectada en formato indice (0-9000)
                # filtrado solo para clases vehiculo
                if cls_id in VEHICLE_CLASSES:
                    # Datos básicos
                    nombre_clase = model.names[cls_id] # Etiqueta textual
                    conf_vehiculo = float(box.conf[0]) # Confianza de la caja
                    coords = list(map(int, box.xyxy[0])) # coordenadas de la caja [x1, y1, x2, y2]
                    x1, y1, x2, y2 = coords

                    # Recortamos el vehículo de la imagen original (region PIL)
                    recorte_vehiculo = img.crop((x1, y1, x2, y2))

                    # Color promedio del vehiculo
                    color_hex = obtener_color_hex(recorte_vehiculo)

                    # Lectura de Placa (OCR)
                    datos_placa = None
                    if recorte_vehiculo.width > 50 and recorte_vehiculo.height > 30:
                        datos_placa = leer_placa_vehiculo(recorte_vehiculo)

                    if datos_placa:
                        vehiculos_con_placa += 1

                    # Creacion de Bounding Boxes vehiculo a vehiculo
                    draw.rectangle(coords, outline="lime", width=2) # 1. Dibujar el Bounding Box (línea verde)
                    placa_texto = datos_placa["texto"] if datos_placa else "Sin Placa" # 2. Texto a mostrar en la caja
                    texto_display = f"#{vehiculo_index} {nombre_clase} | {placa_texto}" # Se usa el índice del vehículo para referenciarlo en el JSON
                    text_bbox = draw.textbbox((x1, y1), texto_display, font=font) # 3. Dibujar fondo para el texto (para mejor legibilidad)
                    draw.rectangle([x1, text_bbox[1], text_bbox[2] + 5, text_bbox[3] + 5], fill="lime") # 4. Dibujar el texto (color negro)
                    draw.text((x1 + 2, y1 + 2), texto_display, fill="black", font=font)


                    # Armamos el objeto de entrega
                    detecciones.append({
                        "id_vehiculo": vehiculo_index,
                        "tipo": nombre_clase,
                        "confianza_vehiculo": round(conf_vehiculo, 2),
                        "color_hex": color_hex,
                        "placa": datos_placa,
                        "posicion": coords
                    })

                    vehiculo_index += 1

        # PREPARAR IMAGEN BASE64
        buffer = io.BytesIO() # Guardamos la imagen con los Bounding Boxes propios en un buffer de memoria
        img_annotated.save(buffer, format="JPEG")
        imagen_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8") # Codificamos el buffer a base64 (string) Así se reciben las imágenes en el API

        # Devuelve un JSON con la información detectada y la imagen anotada
        return jsonify({
            "meta": {"ancho": width, "alto": height},
            "total_vehiculos": len(detecciones),
            "vehiculos_con_placa_leida": vehiculos_con_placa,
            "detalles": detecciones,
            "imagen_analizada_base64": imagen_base64,
            "mimeType": "image/jpeg"
        })

# si algo falla maneja las excepciones con mensaje 500 (web, problema de servidor)
    except Exception as e:
        # Se imprime el error en la consola del servidor para depuración
        print(f"Error en pixmind_worker_vehicle: {e}")
        return jsonify({"error": str(e)}), 500

# Función LocalTunnel y Arranque
def run_localtunnel():
    cmd = ENVlocaltunnel # Se ejecuta con el nombre del subdominio definido en env.
    subprocess.call(cmd, shell=True)

# Ejecutar el localtunnel en un hilo aparte
threading.Thread(target=run_localtunnel).start()
print("Servidor iniciando en puerto 5000...")
app.run(port=5000) # Bloquea el servicio en el puerto 5000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
changed 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

Cargando YOLOv8 Nano...
Cargando EasyOCR...
Servidor iniciando en puerto 5000...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 04:35:45] "POST /pixmindVehicle HTTP/1.1" 200 -


## NoBG
remueve fondos usando AI

In [ ]:
# Instalamos la librería mágica
!pip install flask rembg pillow onnxruntime --quiet
!npm install -g localtunnel

from flask import Flask, request, send_file
from rembg import remove
from PIL import Image
import io
import threading
import subprocess


app = Flask(__name__)

@app.route("/pixmindNoBG", methods=["POST"])
def worker_nobg():
    if "imagen" not in request.files:
        return "Falta imagen", 400

    # 1. Leer imagen
    archivo = request.files['imagen']
    input_img = Image.open(archivo.stream)

    # 2. Magia: Remover fondo
    # Esto devuelve una imagen PIL en formato RGBA (transparente)
    output_img = remove(input_img)

    # 3. Preparar para enviar (Guardar en RAM)
    img_io = io.BytesIO()
    output_img.save(img_io, 'PNG')
    img_io.seek(0)

    # 4. Devolver la imagen directamente
    return send_file(img_io, mimetype='image/png')

# Función para ejecutar localtunnel en un hilo aparte
def run_localtunnel():
    # El puerto debe coincidir con el de Flask
    cmd = ENVlocaltunnel
    subprocess.call(cmd, shell=True)

# Ejecutar localtunnel en segundo plano
threading.Thread(target=run_localtunnel).start()

# Ejecutar Flask
app.run(port=5000)

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹ * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


## Art
Transforma una foto en una obra de un artista famoso

In [ ]:
# Instalamos TensorFlow y Hub
!pip install flask flask-ngrok tensorflow tensorflow_hub pillow --quiet
!npm install -g localtunnel

import tensorflow as tf
import tensorflow_hub as hub
from flask import Flask, request, send_file
from PIL import Image
import numpy as np
import io
import threading
import subprocess

print("Cargando modelo artístico...")
# Modelo capaz de mezclar cualquier estilo con cualquier contenido
hub_model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')

def tensor_to_image(tensor):
    tensor = tensor * 255
    tensor = np.array(tensor, dtype=np.uint8)
    if np.ndim(tensor) > 3:
        tensor = tensor[0]
    return Image.fromarray(tensor)

def load_img(uploaded_file):
    # Preprocesamiento para que TF lo entienda
    img = Image.open(uploaded_file).convert('RGB')
    # Redimensionamos un poco para que no explote la memoria (max 512px)
    max_dim = 512
    img.thumbnail((max_dim, max_dim))

    img_np = np.array(img) / 255.0
    img_tensor = tf.convert_to_tensor(img_np, dtype=tf.float32)
    return img_tensor[tf.newaxis, ...]

app = Flask(__name__)

@app.route("/pixmindArt", methods=["POST"])
def worker_art():
    # Necesitamos dos imágenes: 'contenido' (tu foto) y 'estilo' (la pintura)
    if "contenido" not in request.files or "estilo" not in request.files:
        return "Faltan archivos (contenido y estilo)", 400

    content_image = load_img(request.files['contenido'])
    style_image = load_img(request.files['estilo'])

    # Magia: Inferencia de estilo
    stylized_image = hub_model(tf.constant(content_image), tf.constant(style_image))[0]

    # Convertir resultado a imagen enviarle
    final_img = tensor_to_image(stylized_image)

    img_io = io.BytesIO()
    final_img.save(img_io, 'JPEG', quality=90)
    img_io.seek(0)

    return send_file(img_io, mimetype='image/jpeg')

def run_localtunnel():
    subprocess.call(ENVlocaltunnel, shell=True)

threading.Thread(target=run_localtunnel).start()
app.run(port=5000)

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼Cargando modelo artístico...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:04:55] "POST /pixmindArt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:05:40] "POST /pixmindArt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:07:33] "POST /pixmindArt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:09:05] "POST /pixmindArt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:09:22] "POST /pixmindArt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:09:56] "POST /pixmindArt HTTP/1.1" 200 -


## Roads
Detecta y clasifica baches en la via

In [ ]:
from flask import Flask, request, jsonify
from ultralytics import YOLO
from PIL import Image, ImageDraw
import threading, subprocess, io, base64

app = Flask(__name__)

# Modelo YOLO entrenado para baches
model = YOLO('yolov8n.pt')

@app.route("/pixmindRoads", methods=["POST"])
def worker_roads():
    if "imagen" not in request.files:
        return "Falta imagen", 400

    img = Image.open(request.files['imagen'].stream).convert("RGB")
    results = model(img, conf=0.3, verbose=False)

    daños_viales = []

    # Crear objeto para dibujar
    draw = ImageDraw.Draw(img)

    for r in results:
        for box in r.boxes:
            coords = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
            confianza = float(box.conf[0])

            # Determinar gravedad
            ancho = coords[2] - coords[0]
            gravedad = "ALTA" if ancho > 100 else "MEDIA"

            daños_viales.append({
                "tipo": "bache",
                "gravedad": gravedad,
                "confianza": round(confianza, 2),
                "coordenadas_imagen": coords
            })

            # Dibujar rectángulo rojo sobre la imagen
            draw.rectangle(coords, outline="red", width=3)

    # Convertir imagen anotada a base64
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

    return jsonify({
        "estado_via": "CRITICO" if len(daños_viales) > 2 else "NORMAL",
        "total_incidentes": len(daños_viales),
        "detalles": daños_viales,
        "imagen_anotada_base64": img_base64  # <-- Aquí la imagen anotada
    })

def run_localtunnel():
    subprocess.call(ENVlocaltunnel, shell=True)

threading.Thread(target=run_localtunnel).start()
app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:47:27] "POST /pixmindRoads HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:54:28] "POST /pixmindVehicle HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:54:42] "POST /pixmindVehicle HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:55:38] "POST /pixmindVehicle HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:56:03] "POST /pixmindVehicle HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:58:23] "POST /pixmindVehicle HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 00:58:48] "POST /pixmindRoads HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:00:29] "POST /pixmindVehicle HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Nov/2025 01:02:03] "POST /pixmindRoads HTTP/1.1" 200 -


## NoParking
Detecta en un espacio de la iamgen si hay presencia de un vehículo
- imagen
- zona de deteccion [x1,y1,x2,y2] ej: [300, 580, 1550, 1050]

In [ ]:
from flask import Flask, request, jsonify
from ultralytics import YOLO
from PIL import Image, ImageDraw # Importa ImageDraw para dibujar la zona
import threading
import subprocess
import json
import io     # Nuevo: Para manejar la imagen en memoria
import base64 # Nuevo: Para codificar la imagen a base64

app = Flask(__name__)
model = YOLO('yolov8n.pt')

VEHICULOS = [1, 2, 3, 5, 7] # Bicicleta, Car, Motorcycle, Bus, Truck

def punto_en_zona(x_centro, y_centro, zona):
    """
    Verifica si el punto central está dentro del rectángulo de la zona.
    zona expected format: [x_min, y_min, x_max, y_max]
    """
    zx1, zy1, zx2, zy2 = map(float, zona)
    return (zx1 < x_centro < zx2) and (zy1 < y_centro < zy2)

@app.route("/pixmindNoParking", methods=["POST"])
def worker_parking():
    # --- 1. VALIDACIÓN DE PARÁMETROS DE ENTRADA ---
    if "imagen" not in request.files:
        return jsonify({"error": "Falta el archivo 'imagen' en el form-data", "success": False}), 400

    if "zona" not in request.form:
         return jsonify({"error": "Falta el parámetro 'zona'. Debe enviar un JSON string como '[x1,y1,x2,y2]'", "success": False}), 400

    try:
        zona_str = request.form['zona']
        zona_prohibida = json.loads(zona_str)

        if not isinstance(zona_prohibida, list) or len(zona_prohibida) != 4:
             raise ValueError("El formato debe ser una lista de 4 coordenadas [x1, y1, x2, y2]")

        zona_prohibida = [float(c) for c in zona_prohibida]

    except json.JSONDecodeError:
         return jsonify({"error": "El parámetro 'zona' no es un JSON válido", "success": False}), 400
    except ValueError as e:
         return jsonify({"error": str(e), "success": False}), 400
    except Exception as e:
         return jsonify({"error": f"Error inesperado procesando la zona: {e}", "success": False}), 400

    # --- 2. PROCESAMIENTO DE IMAGEN (YOLO) ---
    try:
        img_file = request.files['imagen'].stream
        img = Image.open(img_file).convert("RGB") # Aseguramos que la imagen está en RGB para dibujar

        results = model(img, conf=0.3, verbose=False)

        infracciones = []

        # Obtenemos la imagen con las detecciones dibujadas por YOLO
        # results[0].plot() devuelve un array numpy de la imagen con las anotaciones
        img_result_np = results[0].plot()
        # Convertimos el array numpy de vuelta a una imagen PIL para seguir dibujando y guardar
        img_with_detections = Image.fromarray(img_result_np[..., ::-1]) # [..., ::-1] para convertir de BGR a RGB

        # --- DIBUJAR LA ZONA PROHIBIDA EN LA IMAGEN ---
        draw = ImageDraw.Draw(img_with_detections)
        # Convertimos a int para Pillow
        zx1, zy1, zx2, zy2 = map(int, zona_prohibida)

        # Dibuja un rectángulo rojo semi-transparente para la zona
        draw.rectangle([zx1, zy1, zx2, zy2], outline="red", width=3) # Borde rojo
        # Opcional: Para rellenar con color semi-transparente, Pillow es un poco limitado directamente.
        # Una forma sería crear una capa nueva y fusionarla. Por ahora, solo el borde es más simple.


        for r in results:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id in VEHICULOS:
                    coords = box.xyxy[0].tolist()

                    cx = (coords[0] + coords[2]) / 2
                    cy = (coords[1] + coords[3]) / 2

                    if punto_en_zona(cx, cy, zona_prohibida):
                        confianza = float(box.conf[0])
                        infracciones.append({
                            "vehiculo": model.names[cls_id],
                            "confianza": round(confianza, 2),
                            "alerta": "VEHICULO EN ZONA PROHIBIDA",
                            "posicion_centro_detectado": [int(cx), int(cy)],
                            "bounding_box": [int(c) for c in coords]
                        })

        # --- CODIFICAR LA IMAGEN RESULTANTE A BASE64 ---
        buffered = io.BytesIO()
        img_with_detections.save(buffered, format="JPEG") # Guarda la imagen en el buffer
        img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8") # Codifica a base64

        return jsonify({
            "success": True,
            "zona_monitorizada": zona_prohibida,
            "hay_infraccion": len(infracciones) > 0,
            "total_infracciones": len(infracciones),
            "detalles": infracciones,
            "imagen_analizada_base64": img_base64 # <-- Nuevo campo con la imagen
        })
    except Exception as e:
         return jsonify({"error": f"Error al procesar la imagen: {e}", "success": False}), 500

# --- Configuración y Ejecución del Túnel y la App ---
def run_localtunnel():
    try:
        print(f"Iniciando localtunnel con el comando: {ENVlocaltunnel}")
        process = subprocess.run(ENVlocaltunnel, shell=True, capture_output=True, text=True)
        print("Salida de localtunnel:\n", process.stdout)
        if process.stderr:
            print("Errores de localtunnel:\n", process.stderr)
        if process.returncode != 0:
            print(f"Localtunnel terminó con código de error {process.returncode}")
    except Exception as e:
        print(f"Error al iniciar localtunnel: {e}")

if __name__ == '__main__':
    threading.Thread(target=run_localtunnel).start()
    print("Iniciando la aplicación Flask en http://127.0.0.1:5000/")
    app.run(port=5000, debug=True, use_reloader=False)

Iniciando localtunnel con el comando: lt --port 5000 --subdomain pixmindworkersIniciando la aplicación Flask en http://127.0.0.1:5000/

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/Nov/2025 17:37:26] "POST /pixmindNoParking HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Nov/2025 17:38:27] "POST /pixmindNoParking HTTP/1.1" 200 -


Salida de localtunnel:
 your url is: https://pixmindworkers.loca.lt

Localtunnel terminó con código de error -2
